In [8]:
!pip install pillow python-chess google-generativeai requests


In [11]:
# gemini_temporal_reasoning.py
# Track B: Temporal Reasoning + Stockfish eval + Gemini next move prediction + reply prediction

import time
import os
import io
import requests
import google.generativeai as genai
from PIL import Image
import chess  # python-chess library


# --- Gemini Setup ---
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Missing GOOGLE_API_KEY environment variable.")
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# --- Stockfish API ---
STOCKFISH_API = "https://stockfish.online/api/s/v2.php"

def query_stockfish(fen: str) -> dict:
    """Query Stockfish.online API with a FEN string."""
    params = {"fen": fen, "depth": 15}
    try:
        resp = requests.get(STOCKFISH_API, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return {
            "stockfish_eval": data.get("evaluation", "N/A"),
            "stockfish_best_move": data.get("bestmove", "N/A"),
            "stockfish_ponder": data.get("ponder", "N/A")
        }
    except Exception as e:
        print(f"Stockfish query failed: {e}")
        return {"stockfish_eval": "N/A", "stockfish_best_move": "N/A", "stockfish_ponder": "N/A"}

# --- Gemini Move Analyzer ---
def analyze_move(img1: Image.Image, img2: Image.Image) -> dict:
    """Compare two chessboard frames and detect the move."""
    prompt = (
        "Compare these two chessboard images. "
        "Identify the move in coordinate notation (e.g., e2-e4). "
        "If no move is detected, respond 'no move'."
    )

    start = time.time()
    response = model.generate_content([prompt, img1, img2])
    latency = round((time.time() - start) * 1000, 2)

    move = response.text.strip().replace(" ", "")
    return {"move": move, "latency_ms": latency}

# --- Gemini Next Move Predictor ---
def predict_next_move(img_last: Image.Image) -> str:
    """Ask Gemini to predict the next most likely move from the final board frame."""
    prompt = (
        "Look at this chessboard position. "
        "Predict the next most likely move in coordinate notation only (e.g., e7-e5). "
        "Do not explain or add extra text."
    )
    response = model.generate_content([prompt, img_last])
    return response.text.strip()

# --- Convert moves to FEN ---
def moves_to_fen(moves: list, start_fen: str = chess.STARTING_FEN) -> str:
    """Apply a sequence of moves to a starting FEN."""
    board = chess.Board(start_fen)
    for move in moves:
        try:
            uci = move.lower().replace("-", "").replace(" ", "")
            if len(uci) == 4:
                board.push_uci(uci)
            else:
                board.push_san(move)
        except Exception as e:
            print(f"Skipping invalid move '{move}': {e}")
    return board.fen()

# --- Main Evaluation ---
def evaluate_sequence(image_paths: list, ground_truth_moves: list):
    """Run Gemini + Stockfish reasoning pipeline with local image files."""
    detected_moves, latencies = [], []

    # Step 1: Move detection
    for i in range(len(image_paths) - 1):
        img1 = Image.open(image_paths[i])
        img2 = Image.open(image_paths[i+1])
        result = analyze_move(img1, img2)
        detected_moves.append(result["move"])
        latencies.append(result["latency_ms"])

    # Step 2: Compute metrics
    correct = 0
    for g, d in zip(ground_truth_moves, detected_moves):
        if g.lower().replace("-", "") == d.lower().replace("-", ""):
            correct += 1
    seq_accuracy = round((correct / len(ground_truth_moves)) * 100, 2)
    mae = abs(len(detected_moves) - len(ground_truth_moves))
    avg_latency = round(sum(latencies) / len(latencies), 2)

    # Step 3: FEN + Stockfish + Gemini prediction
    final_fen = moves_to_fen(detected_moves)
    sf_final = query_stockfish(final_fen)
    gemini_best = predict_next_move(Image.open(image_paths[-1]))
    gemini_fen = moves_to_fen(detected_moves + [gemini_best])
    sf_after_gemini = query_stockfish(gemini_fen)

    return {
        "predicted_moves": detected_moves,
        "ground_truth": ground_truth_moves,
        "sequence_accuracy": seq_accuracy,
        "event_mae": mae,
        "avg_latency_ms": avg_latency,
        "final_fen": final_fen,
        "stockfish_eval": sf_final["stockfish_eval"],
        "stockfish_best_move": sf_final["stockfish_best_move"],
        "gemini_best_move": gemini_best,
        "gemini_eval": sf_after_gemini["stockfish_eval"],
        "stockfish_reply_to_gemini": sf_after_gemini["stockfish_best_move"]
    }

# --- Optional: run directly ---
if __name__ == "__main__":
    # Example test (replace with your actual image paths)
    image_files = [
        "Board State 1.png",
        "Board State 2.png",
        "Board State 3.png",
        "Board State 4.png"
    ]
    ground_truth = ["b1-c3", "g8-f6", "e2-e4"]

    results = evaluate_sequence(image_files, ground_truth)
    print("\n=== Temporal Reasoning Results ===")
    for k, v in results.items():
        print(f"{k}: {v}")



=== Temporal Reasoning Results ===
predicted_moves: ['b1-c3', 'g8-f6', 'e2-e4']
ground_truth: ['b1-c3', 'g8-f6', 'e2-e4']
sequence_accuracy: 100.0
event_mae: 0
avg_latency_ms: 7265.99
final_fen: rnbqkb1r/pppppppp/5n2/8/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 0 2
stockfish_eval: 0.05
stockfish_best_move: bestmove e7e5 ponder g1f3
gemini_best_move: d7-d5
gemini_eval: 0.99
stockfish_reply_to_gemini: bestmove e4d5 ponder f6d5
